In [1]:
# installing packages
pip install transformers torch nltk faiss-cpu streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9

In [2]:
# Step 2: Loading a Small Language Model Locally
# Since we’re avoiding GPT APIs calls, we can use a small, open-source model like GPT-2 for content generation.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Loading GPT-2 (small model to run on CPU)
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
# Step 3: Implementing Sentiment-Based Content Generation


In [5]:
# We first generate text without sentiment control and then filter/refine it using sentiment analysis.

In [10]:
import nltk
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from nltk.sentiment import SentimentIntensityAnalyzer

# Downloading and initialize sentiment analyzer
nltk.download("vader_lexicon")
sia = SentimentIntensityAnalyzer()

# Loading GPT-2 model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Ensuring tokenizer has a padding token
tokenizer.pad_token = tokenizer.eos_token

def generate_content(prompt, sentiment, max_retries=5, retries=0):

    if retries >= max_retries:
        return "Couldn't generate suitable content after multiple attempts."

    # Modify the prompt to encourage the desired sentiment
    if sentiment == "positive":
        prompt = f"{prompt} This made me feel incredibly happy and grateful."
    elif sentiment == "negative":
        prompt = f"{prompt} It was an awful experience, and I felt deeply upset."
    elif sentiment == "neutral":
        prompt = f"{prompt} It was just another ordinary day."

    # Encode input and create attention mask
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    attention_mask = torch.ones_like(input_ids)

    # Generate text with anti-repetition strategies
    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=100,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id,  # Avoid padding issues
        repetition_penalty=1.2,  # Discourages repeated phrases
        no_repeat_ngram_size=3,  # Prevents repeating trigrams
        temperature=0.7,  # Adds randomness to avoid monotony
        top_k=50,  # Filters unlikely words
        top_p=0.95  # Nucleus sampling for diversity
    )

    # Decode generated text
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Analyze sentiment
    sentiment_score = sia.polarity_scores(generated_text)["compound"]

    # Define sentiment thresholds for better flexibility
    if sentiment == "positive" and sentiment_score < 0.2:
        return generate_content(prompt, sentiment, max_retries, retries + 1)
    elif sentiment == "negative" and sentiment_score > -0.2:
        return generate_content(prompt, sentiment, max_retries, retries + 1)
    elif sentiment == "neutral" and (sentiment_score > 0.2 or sentiment_score < -0.2):
        return generate_content(prompt, sentiment, max_retries, retries + 1)

    return generated_text

prompt = "The day was going great until"
sentiment = "negative"  # Choose: positive, negative, neutral

generated_text = generate_content(prompt, sentiment)
print("Generated Text:", generated_text)


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Generated Text: The day was going great until It was an awful experience, and I felt deeply upset.
"I had to go back home because of the pain that it caused me."


In [11]:
prompt = "The day was not going great until"
sentiment = "positive"  # Choose: positive, negative, neutral

generated_text = generate_content(prompt, sentiment)
print("Generated Text:", generated_text)

Generated Text: The day was not going great until This made me feel incredibly happy and grateful. I had been thinking about this for a while, but it just didn't seem right to be so excited when the next morning came around with my first big smile on her face!

I'm sure she's already got some of that in mind as well…
